# 03 · EDA Educación

Dos fuentes de la Secretaría de Educación (SED) en `data/raw/EDUCACION/`:
- `colegios122025.gpkg` — sedes educativas (capa `colegios_122025`, ~2.211 puntos, EPSG:3857).
- `matricula_total_colegios_oficiales.gpkg` — matrícula total por sede oficial (capa
  `matriculatotal_042025`, 747 filas, 55 columnas, EPSG:3857).

**Indicadores objetivo**: EDU-01 (cobertura educativa por localidad), EDU-02 (matrícula oficial
por localidad). Ambas usan `COD_LOCA` (código 1-20) como clave territorial.

## 0. Configuración

In [ ]:
import sys, os, re, json, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
warnings.filterwarnings("ignore")

if (Path("../..") / "src" / "eda").exists():
    ROOT = Path("../..").resolve()
elif (Path("..") / "src" / "eda").exists():
    ROOT = Path("..").resolve()
elif (Path(".") / "src" / "eda").exists():
    ROOT = Path(".").resolve()
else:
    ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw"
MR_PATH = RAW_DIR / "INFRAESTRUCTURA_ESPACIO_PUBLICO" / "gpkg_mr_v03.26" / "gpkg_mr_v03.26.gpkg"
REPORTS = ROOT / "reports" / "eda"
PERFILES = REPORTS / "perfiles"
CACHE = REPORTS / "cache"
TIEMPOS = REPORTS / "tiempos"
for _d in (REPORTS, PERFILES, CACHE, TIEMPOS):
    _d.mkdir(parents=True, exist_ok=True)

SMOKE = os.environ.get("EDA_SMOKE") == "1"
print("ROOT:", ROOT, "| SMOKE:", SMOKE)

import src.eda as eda
from src.eda.explore import explorar_dataset

_SECTION_T = {}

def t0(nombre):
    _SECTION_T[nombre] = time.time()

def t1(nombre):
    _SECTION_T[nombre] = time.time() - _SECTION_T[nombre]

def guardar_tiempos(archivo):
    df = pd.DataFrame([{"seccion": k, "segundos": round(v, 1)} for k, v in _SECTION_T.items()])
    df.to_csv(TIEMPOS / archivo, index=False)
    print(f"tiempos guardados: {archivo} ({len(df)} secciones)")

## 0.1 Fuentes del sector en el catálogo

In [ ]:
cat = eda.load_catalog()
sec = cat[cat["indicadores"].astype(str).str.contains("EDU", case=False, na=False)]
display(sec[["id", "nombre", "archivo", "temporalidad", "indicadores", "valor_publico"]])

## 1. Sedes educativas

### Sedes educativas — SED

In [ ]:
t0('colegios')
SPEC = {
    'id': 'colegios_122025',
    'titulo': 'Sedes educativas — SED',
    'path': 'data/raw/EDUCACION/colegios122025.gpkg',
    'capa': 'colegios_122025',
    'origen': 'Secretaría de Educación del Distrito (SED)',
    'corte': 'Dic 2025 (nombre del archivo)',
    'valor_publico': 'Oferta educativa por localidad: sedes, sector, calendario, carácter, estrato',
    'indicadores': 'EDU-01 (sedes por localidad)',
    'notas': 'COD_LOCA es código numérico 1-20: se mapea a nombre con localidad_de_codigo. 30 columnas.',
}
RES = explorar_dataset(SPEC, RAW_DIR, PERFILES, smoke=SMOKE)
t1('colegios')

## 2. Matrícula total oficial

### Matrícula total por sede oficial — SED

In [ ]:
t0('matricula')
SPEC = {
    'id': 'matricula_oficial_042025',
    'titulo': 'Matrícula total por sede oficial — SED',
    'path': 'data/raw/EDUCACION/matricula_total_colegios_oficiales.gpkg',
    'capa': 'matriculatotal_042025',
    'origen': 'Secretaría de Educación del Distrito (SED)',
    'corte': 'Abr 2025 (nombre del archivo)',
    'valor_publico': 'Matrícula oficial por localidad, con desagregación por discapacidad, etnia y talentos',
    'indicadores': 'EDU-02 (matrícula por localidad)',
    'notas': '55 columnas: TMATRIC_GE es la matrícula general; columnas DA_*/DI_*/DV_*/DIS_*/T_* agregan condiciones de discapacidad, etnia y talentos. COD_LOCA numérico 1-20.',
}
RES = explorar_dataset(SPEC, RAW_DIR, PERFILES, smoke=SMOKE)
t1('matricula')

## 3. Análisis de cobertura educativa

### 3.1 Sedes por localidad y mapa

In [ ]:
from src.eda.profiling import localidad_de_codigo
from src.eda.spatial import load_loca
col = gpd.read_file(str(RAW_DIR / "EDUCACION" / "colegios122025.gpkg"), layer="colegios_122025")
col["LOCALIDAD"] = col["COD_LOCA"].map(localidad_de_codigo)
conteo = col.groupby("LOCALIDAD").size().sort_values(ascending=False)
conteo.to_csv(REPORTS / "educacion_colegios_por_localidad.csv", index=False)
display(conteo.rename("sedes"))
print("Total sedes:", len(col), "| localidades con sedes:", conteo.notna().sum())
conteo.plot(kind="bar", figsize=(11, 4), title="Sedes educativas por localidad", color="#4c72b0")
plt.ylabel("sedes")
plt.show()

fig, ax = plt.subplots(figsize=(9, 9))
loca = load_loca(MR_PATH)
loca.plot(ax=ax, color="#f0f0e8", edgecolor="#9a9a8a", linewidth=0.4)
col.to_crs(loca.crs).plot(ax=ax, color="#2ca02c", markersize=3, alpha=0.6)
ax.set_title(f"Sedes educativas ({len(col)})")
ax.set_axis_off()
plt.show()

### 3.2 Matrícula oficial por localidad

In [ ]:
mat = gpd.read_file(str(RAW_DIR / "EDUCACION" / "matricula_total_colegios_oficiales.gpkg"), layer="matriculatotal_042025")
mat["LOCALIDAD"] = mat["COD_LOCA"].map(localidad_de_codigo)
tot = mat.groupby("LOCALIDAD")["TMATRIC_GE"].sum().sort_values(ascending=False)
tot.to_csv(REPORTS / "educacion_matricula_por_localidad.csv", index=False)
print("Colegios con matrícula:", len(mat), "| matrícula total:", f"{mat['TMATRIC_GE'].sum():,.0f}")
tot.plot(kind="bar", figsize=(11, 4), title="Matrícula oficial por localidad", color="#55a868")
plt.ylabel("estudiantes")
plt.show()

### 3.3 Matrícula: discapacidad, etnia y talentos

In [ ]:
disc = [c for c in mat.columns if c.startswith(("DA_", "DI_", "DV_", "DIS_", "T_")) or c in ("SORDOCEGUE", "OTRA")]
et_pct = mat["TOT_EST_ET"].sum() / mat["TMATRIC_GE"].sum()
disc_pct = mat[disc].sum().sum() / mat["TMATRIC_GE"].sum()
print(f"Estudiantes con condición de discapacidad: {disc_pct:.2%}")
print(f"Población étnica (afro, indígena, etc.): {et_pct:.2%}")
tal = mat[["TE_ACTFISI", "TE_ARTES", "TE_CNATUR", "TE_CSOCIAL", "TE_LDSOCIA", "TE_TECNOLO"]].sum()
tal.plot(kind="bar", figsize=(8, 4), title="Estudiantes por talento excepcional", color="#8172b3")
plt.ylabel("estudiantes")
plt.show()

## 4. Indicadores del sector

In [ ]:
sts = eda.indicator_status()
sts_sec = sts[sts["indicador"].str.startswith("EDU", na=False)]
display(sts_sec[["indicador", "dimension", "estado", "que_falta"]])
sts_sec.to_csv(REPORTS / "indicadores_educacion.csv", index=False)
guardar_tiempos("03_eda_educacion.csv")
print("Secciones del notebook:", list(_SECTION_T.keys()))